In [ ]:
#curl -s https://raw.githubusercontent.com/aws/sagemaker-python-sdk/master/sagemaker-core/src/sagemaker/core/image_uri_config/xgboost.json \
#  | jq -r '.inference.versions["1.7-1"] | "\(.registries["us-west-2"]).dkr.ecr.us-west-2.amazonaws.com/\(.repository):1.7-1"'

In [ ]:
#683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1

1. we keep adding models to the S3 bucket correct , so do we need to manually change here to update to new model?

2. but with new model we need to test first right before pushing to prod how does that fit here?



In [2]:

import boto3
import configparser
import requests
import numpy as np
import pandas as pd
from io import StringIO
from pathlib import Path

In [3]:
endpoint_name = "hotel-cancel"
repo = "https://raw.githubusercontent.com/pulkit-scaler/mlops-model-registry/main"

session = boto3.Session()

if session.get_credentials() is None:
    config_path = Path("aws_config.cfg")

    if not config_path.exists():
        raise FileNotFoundError(f"AWS credentials not found. Missing {config_path}")

    config = configparser.ConfigParser()
    config.read(config_path)

    aws_config = config["AWS"]

    session = boto3.Session(
        aws_access_key_id=aws_config["aws_access_key_id"],
        aws_secret_access_key=aws_config["aws_secret_access_key"],
        aws_session_token=aws_config.get("aws_session_token"),
        region_name=aws_config.get("region_name", session.region_name)
    )


# Version 1

In [4]:

sagemaker = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")

live_response = requests.get(f"{repo}/data/live.csv", timeout=30)
labels_response = requests.get(f"{repo}/data/live_labels.csv", timeout=30)

live_response.raise_for_status()
labels_response.raise_for_status()

live_csv = live_response.content
labels = pd.read_csv(
    StringIO(labels_response.text), header=None
).iloc[:, 0].to_numpy(dtype=int)

components = sagemaker.list_inference_components(
    EndpointNameEquals=endpoint_name,
    StatusEquals="InService"
)["InferenceComponents"]

if len(components) != 1:
    raise ValueError(
        f"Expected one active inference component, found {len(components)}: "
        f"{[c['InferenceComponentName'] for c in components]}"
    )

ic_name = components[0]["InferenceComponentName"]
print(f"Component: {ic_name}")


Component: hotel-cancellation-1794846529212-20261116-1628570


In [5]:
#hotel-cancellation-1794846529212-20261116-1628570


response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    ContentType="text/csv",
    Body=live_csv
)

prediction_text = response["Body"].read().decode("utf-8")
predictions = np.fromstring(
    prediction_text.replace(",", " "), sep=" "
)

if len(predictions) != len(labels):
    raise ValueError(
        f"Prediction count ({len(predictions)}) "
        f"does not match label count ({len(labels)})"
    )

print("First three answers:", predictions[:3])

flagged = predictions > 0.5
cancelled = labels == 1

print("Bookings:", len(labels))
print("Really cancelled:", np.sum(cancelled))
print("Flagged:", np.sum(flagged))
print("Flagged but guest came:", np.sum(flagged & ~cancelled))

First three answers: [0.20676228 0.69760454 0.36289686]
Bookings: 1000
Really cancelled: 298
Flagged: 239
Flagged but guest came: 78


# Version 2

In [6]:

sagemaker = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")

live_response = requests.get(f"{repo}/data/live.csv", timeout=30)
labels_response = requests.get(f"{repo}/data/live_labels.csv", timeout=30)

live_response.raise_for_status()
labels_response.raise_for_status()

live_csv = live_response.content
labels = pd.read_csv(
    StringIO(labels_response.text), header=None
).iloc[:, 0].to_numpy(dtype=int)

components = sagemaker.list_inference_components(
    EndpointNameEquals=endpoint_name,
    StatusEquals="InService"
)["InferenceComponents"]

if len(components) != 1:
    raise ValueError(
        f"Expected one active inference component, found {len(components)}: "
        f"{[c['InferenceComponentName'] for c in components]}"
    )

ic_name = components[0]["InferenceComponentName"]
print(f"Component: {ic_name}")


Component: hotel-cancellation-1794848029935-20261116-1653530


In [7]:

#hotel-cancellation-1794846529212-20261116-1628570


response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    ContentType="text/csv",
    Body=live_csv
)

prediction_text = response["Body"].read().decode("utf-8")
predictions = np.fromstring(
    prediction_text.replace(",", " "), sep=" "
)

if len(predictions) != len(labels):
    raise ValueError(
        f"Prediction count ({len(predictions)}) "
        f"does not match label count ({len(labels)})"
    )

print("First three answers:", predictions[:3])

flagged = predictions > 0.5
cancelled = labels == 1

print("Bookings:", len(labels))
print("Really cancelled:", np.sum(cancelled))
print("Flagged:", np.sum(flagged))
print("Flagged but guest came:", np.sum(flagged & ~cancelled))

First three answers: [0.38796833 0.85528272 0.57388157]
Bookings: 1000
Really cancelled: 298
Flagged: 513
Flagged but guest came: 241
